<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/Control%20Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Test


In [29]:
# @title Env
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)
print(f"\n✅ Installed EnergyPlus")

!pip install -q control
print(f"\n✅ Installed Control")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+5

✅ Installed EnergyPlus

✅ Installed Control


In [30]:
import types, datetime, requests, io, os, gc
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from eplus.core import EPlusUtil

In [42]:
# @title Setup E+
OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneCAV_MaxTemp.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)

sim.run_dry_run(include_ems_edd=False, reset=True, design_day=True)
print("Dry Run Complete!")


Initialized StateMixin
Initialized EnergyPlus State.
Initialized IDFMixin
Initialized LoggingMixin
Initialized SimulationMixin
Initialized UtilsMixin
Initialized HandlersMixin
Initialized SQLMixin
Initialized ControlMixin
Initialized OccupancyMixin
Initialized ZoneObserverMixin
EnergyPlus state has been reset.
Deleted output directory: /simulation/eplus_out
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/5ZoneCAV_MaxTemp.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
EnergyPlus state has been reset.
EnergyPlus state has been reset.
Dry Run Complete!


In [43]:
# @title CO2 and People Injector

def preload_occupancy_csv(sim_obj, url):
    """
    Downloads the CSV, anchors time to midnight, and forces a perfect 24-hour loop.
    """
    print(f"Downloading CSV from: {url}...")
    try:
        resp = requests.get(url)
        resp.raise_for_status()

        df = pd.read_csv(io.StringIO(resp.text))
        if 'timestamp' not in df.columns:
            raise ValueError("CSV must contain a 'timestamp' column.")

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')

        # 1. Anchor to MIDNIGHT of the first day to prevent timestep offset
        midnight_start = df['timestamp'].iloc[0].normalize()
        df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()

        # 2. Force exactly 24 hours for daily looping to prevent modulo drift
        sim_obj._occ_duration_sec = 86400.0

        # 3. Clean up dataframe
        df = df.set_index('rel_seconds')
        sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])

        zones = list(sim_obj._preloaded_occ_df.columns)
        print(f"Success! Preloaded {len(df)} rows. Loop locked to 24.00 hours.")
        print(f"Detected Source Columns: {zones}")

    except Exception as e:
        print(f"Failed to preload CSV: {e}")
# Load CSV
csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    """
    Lightning-fast runtime handler. Maps actuators on the first tick.
    Reads a single baseline zone from the CSV and uses multipliers,
    ceilings, and clipping to populate other zones synthetically.
    """
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. One-Time Setup: Map Actuators & Define Extrapolation Rules ---
    if not hasattr(self, '_fast_injector_ready'):
        # Ensure the data was preloaded via Step 1
        if not hasattr(self, '_preloaded_occ_df'):
            print("[Injector] ERROR: Data not preloaded. Run preload_occupancy_csv() first.")
            self._fast_injector_ready = False
            return

        # =========================================================
        # CONFIGURATION DICTIONARY: TWEAK YOUR MULTIPLIERS HERE
        # source: The CSV column to read the baseline value from
        # mult: The multiplier applied to the source value
        # min/max: The clipping bounds to enforce physical limits
        # =========================================================
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0,  "min": 0, "max": 5}, # Baseline
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5,  "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4,  "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2,  "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0,  "min": 0, "max": 6},
        }

        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())

        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []

        # Map to EnergyPlus Actuators based on the target zones, NOT just CSV columns
        mapped_count = 0
        for z in target_zones:
            matched_people = [p for p in ep_people_names if z.replace(" ", "").lower() in p.replace(" ", "").lower()]
            handles = []
            for p in matched_people:
                h = self.exchange.get_actuator_handle(state, "People", "Number of People", p)
                if h != -1:
                    handles.append(h)
                    mapped_count += 1
            if handles:
                self._people_handles[z] = handles

        print(f"\n[Injector] Mapped {mapped_count} actuators across {len(self._people_handles)} zones (Extrapolation Active).")

        # Mark exact start time of the simulation
        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

        self._fast_injector_ready = True

    # --- Runtime Safety Check ---
    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec', 0) == 0:
        return

    # --- 2. Calculate Elapsed Time & Loop ---
    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec

    # --- 3. Fast Data Lookup (Forward Fill) ---
    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices) == 0 else valid_indices[-1]
    row = df.loc[target_idx]

    # --- 4. Extrapolate and Inject Values ---
    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if not rule:
            continue

        src_col = rule["source"]
        if src_col in row:
            base_val = float(row[src_col])

            # Apply math: Base * Multiplier -> Round Up -> Clip
            if base_val == 0:
                val = 0.0 # Bypasses math to strictly enforce zero at night
            else:
                calculated = np.ceil(base_val * rule["mult"])
                val = float(np.clip(calculated, rule["min"], rule["max"]))

            # Divide evenly if there are multiple People objects in the same room
            per_actuator = val / len(handles)
            for h in handles:
                self.exchange.set_actuator_value(state, h, per_actuator)

# --- Registration ---
sim.people_injector = types.MethodType(people_injector, sim)

sim.register_handlers("begin", [
    {"method_name": "people_injector"},
])

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

Success! Preloaded 10129 rows. Loop locked to 24.00 hours.
Detected Source Columns: ['SPACE1-1']
Handlers on 'begin' hook: ['people_injector']


['data_logger']

In [45]:
# @title Simulation
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 2),
    timestep_per_hour=4,
    start_day_of_week="Sunday",
)

print("Starting controlled simulation...")
res = sim.run_annual()

if res == 0:
    print("Simulation complete.")
else:
    print("Simulation failed. Check eplusout.err")
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])


EnergyPlus state has been reset.
Starting controlled simulation...
Deleted output file: /simulation/eplus_out/eplusout.err
Deleted output file: /simulation/eplus_out/eplusout.audit
EnergyPlus state has been reset.
[after_hvac] master_data_logger failed: ControlMixin.runtime_get_variable() takes 2 positional arguments but 4 were given
[after_hvac] master_data_logger failed: ControlMixin.runtime_get_variable() takes 2 positional arguments but 4 were given
[after_hvac] master_data_logger failed: ControlMixin.runtime_get_variable() takes 2 positional arguments but 4 were given
[after_hvac] master_data_logger failed: ControlMixin.runtime_get_variable() takes 2 positional arguments but 4 were given
[after_hvac] master_data_logger failed: ControlMixin.runtime_get_variable() takes 2 positional arguments but 4 were given
[after_hvac] master_data_logger failed: ControlMixin.runtime_get_variable() takes 2 positional arguments but 4 were given
[after_hvac] master_data_logger failed: ControlMixin.r